In [1]:
!pip install -q streamlit scikit-learn matplotlib numpy

In [2]:
%%writefile app.py
import streamlit as st
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor

# Dashboard Layout Setup
st.set_page_config(page_title="MCTA 4362: Intelligent Controller", layout="wide")
st.title("🤖 Intelligent ML Controller vs. Classical PID Dashboard")
st.write("**Course Code:** MCTA 4362 Machine Learning | **Project:** Intelligent Controller Design")
st.markdown("---")

# Sidebar for Dynamic Control
st.sidebar.header("🛠️ Simulation Control Center")
setpoint = st.sidebar.slider("Target Motor Speed (Setpoint RPM)", min_value=10, max_value=100, value=60, step=5)
disturbance_time = st.sidebar.slider("Disturbance Injection Time (s)", min_value=1.0, max_value=4.0, value=2.5, step=0.1)
disturbance_load = st.sidebar.slider("Disturbance Load Magnitude", min_value=0.0, max_value=10.0, value=5.0, step=0.5)

st.sidebar.subheader("🧠 Machine Learning Parameters")
hidden_layers = st.sidebar.selectbox("ANN Hidden Layer Structure", options=[(10, 10), (20, 20), (5, 5)], index=0)

# DC Motor Mathematical Plant Simulation Constants
dt = 0.01
time = np.arange(0, 5, dt)
R, L = 1.0, 0.5    # Armature Resistance & Inductance
J, b = 0.01, 0.1   # Rotor Inertia & Viscous Friction
K = 0.01           # Torque Constant

def step_motor(omega, i, V, TL=0.0):
    """Simulates one time-step of the DC Motor differential equations"""
    di = (V - R * i - K * omega) / L * dt
    domega = (K * i - b * omega - TL) / J * dt
    return omega + domega, i + di

@st.cache_resource
def train_ann_controller(_layers):
    """Generates synthetic PID data and trains the Neural Network Controller"""
    X_train, Y_train = [], []
    setpoints_train = [10, 30, 50, 70, 100]

    for sp_train in setpoints_train:
        omega, i = 0.0, 0.0
        eprev, eint = 0.0, 0.0
        for t in time:
            error = sp_train - omega
            eint += error * dt
            ederiv = (error - eprev) / dt
            eprev = error

            # Classical PID Equation
            V = 15.0 * error + 5.0 * eint + 0.1 * ederiv
            V = np.clip(V, -24, 24) # 24V Voltage Saturation

            X_train.append([error, eint, ederiv])
            Y_train.append(V)
            omega, i = step_motor(omega, i, V)

    ann = MLPRegressor(hidden_layer_sizes=_layers, max_iter=1000, random_state=42)
    ann.fit(X_train, Y_train)
    return ann

# Execute Training
ann_controller = train_ann_controller(hidden_layers)

def run_simulation(control_type):
    """Runs a 5-second simulation using either PID or the trained ANN"""
    omega, i = 0.0, 0.0
    eprev, eint = 0.0, 0.0
    history = []

    for t in time:
        error = setpoint - omega
        eint += error * dt
        ederiv = (error - eprev) / dt
        eprev = error

        # Inject load torque disturbance at user-selected time
        TL = disturbance_load if t > disturbance_time else 0.0

        if control_type == "PID":
            V = 15.0 * error + 5.0 * eint + 0.1 * ederiv
        else:
            V = ann_controller.predict([[error, eint, ederiv]])[0]

        V = np.clip(V, -24, 24)
        omega, i = step_motor(omega, i, V, TL)
        history.append(omega)
    return np.array(history)

# Run simulations based on UI choices
pid_results = run_simulation("PID")
ann_results = run_simulation("ANN")

# Calculate Performance Metrics
pid_mse = np.mean((setpoint - pid_results) ** 2)
ann_mse = np.mean((setpoint - ann_results) ** 2)

# --- Render the UI Layout ---
col1, col2 = st.columns([3, 1])

with col1:
    st.subheader("📈 Transient Response and System Behavior")
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(time, np.ones_like(time) * setpoint, "k--", label="Target Velocity (Setpoint)")
    ax.plot(time, pid_results, "b-", label="Classical PID Controller", linewidth=2)
    ax.plot(time, ann_results, "r--", label="Intelligent ANN Controller", linewidth=2)
    ax.axvline(x=disturbance_time, color="g", linestyle=":", label="External Torque Disturbance Applied")
    ax.set_xlabel("Time (seconds)", fontsize=10)
    ax.set_ylabel("Motor Velocity (RPM)", fontsize=10)
    ax.legend(loc="lower right")
    ax.grid(True, linestyle="--", alpha=0.5)
    st.pyplot(fig)

with col2:
    st.subheader("📊 Performance Metrics")
    st.metric(label="Classical PID MSE", value=f"{pid_mse:.2f}")
    st.metric(label="Intelligent ANN MSE", value=f"{ann_mse:.2f}")

    st.markdown("### Evaluation Summary")
    if ann_mse <= pid_mse:
        st.success("✅ The Machine Learning model effectively emulates and matches the controller's target profiles.")
    else:
        st.info("ℹ️ Standard linear PID maintains a rigid edge under low-complexity steady states.")

st.markdown("---")
st.subheader("📋 Core Project Information for Evaluators")
st.markdown("""
- **Dynamic System Plant:** DC Motor Speed Regulation (Electrical & Mechanical state mappings).
- **ML Algorithm:** Artificial Neural Network (ANN Multi-Layer Perceptron Regressor).
- **Features Tracked ($X$):** Tracking Error ($e_t$), Error Integral ($\\int e$), Error Derivative ($\\frac{de}{dt}$).
- **Target Value ($Y$):** Control Armature Voltage ($V$).
""")

Overwriting app.py


In [8]:
# =====================================================================
# MCTA 4362 Machine Learning: Mini Project (Gradio Dashboard Fix)
# =====================================================================

# 1. Install Gradio dependency
print("⏳ Installing stable dashboard environment...")
!pip install -q gradio scikit-learn matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import gradio as gr
from sklearn.neural_network import MLPRegressor

print("🧠 Modeling plant and pre-training the Intelligent ANN Controller...")

# --- Plant Dynamic Models ---
dt = 0.01
time = np.arange(0, 5, dt)
R, L = 1.0, 0.5
J, b = 0.01, 0.1
K = 0.01

def step_motor(omega, i, V, TL=0.0):
    di = (V - R * i - K * omega) / L * dt
    domega = (K * i - b * omega - TL) / J * dt
    return omega + domega, i + di

# --- Pre-training Data Pipeline ---
X_train, Y_train = [], []
setpoints_train = [10, 30, 50, 70, 100]

for sp_train in setpoints_train:
    omega, i = 0.0, 0.0
    eprev, eint = 0.0, 0.0
    for t in time:
        error = sp_train - omega
        eint += error * dt
        ederiv = (error - eprev) / dt
        eprev = error
        V = 15.0 * error + 5.0 * eint + 0.1 * ederiv
        V = np.clip(V, -24, 24)
        X_train.append([error, eint, ederiv])
        Y_train.append(V)
        omega, i = step_motor(omega, i, V)

# Fit Neural Network
ann_controller = MLPRegressor(hidden_layer_sizes=(10, 10), max_iter=1000, random_state=42)
ann_controller.fit(X_train, Y_train)
print("🎯 Training complete! Launching Interface...")

# --- Dashboard Simulation Function ---
def simulate_dashboard(setpoint, disturbance_time, disturbance_load):
    def run_sim(control_type):
        omega, i = 0.0, 0.0
        eprev, eint = 0.0, 0.0
        history = []
        for t in time:
            error = setpoint - omega
            eint += error * dt
            ederiv = (error - eprev) / dt
            eprev = error
            TL = disturbance_load if t > disturbance_time else 0.0

            if control_type == "PID":
                V = 15.0 * error + 5.0 * eint + 0.1 * ederiv
            else:
                V = ann_controller.predict([[error, eint, ederiv]])[0]

            V = np.clip(V, -24, 24)
            omega, i = step_motor(omega, i, V, TL)
            history.append(omega)
        return np.array(history)

    pid_omega = run_sim("PID")
    ann_omega = run_sim("ANN")

    # Calculate Metrics
    pid_mse = np.mean((setpoint - pid_omega) ** 2)
    ann_mse = np.mean((setpoint - ann_omega) ** 2)

    # Generate Output Figure
    fig, ax = plt.subplots(figsize=(10, 4.5), dpi=120)
    ax.plot(time, np.ones_like(time) * setpoint, "k--", label="Setpoint Target", alpha=0.7)
    ax.plot(time, pid_omega, "b-", label="Classical PID Controller", linewidth=2)
    ax.plot(time, ann_omega, "r--", label="Intelligent ANN Controller", linewidth=2)
    ax.axvline(x=disturbance_time, color="g", linestyle=":", linewidth=2, label="Disturbance Applied")
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("Motor Velocity (RPM)")
    ax.legend(loc="lower right")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.set_title("Transient Response Comparison: PID vs. Machine Learning ANN")
    plt.tight_layout()

    metrics_text = f"📊 PERFORMANCE EVALUATION:\n\n• Classical PID MSE: {pid_mse:.2f}\n• Intelligent ANN MSE: {ann_mse:.2f}\n\n"
    if ann_mse <= pid_mse:
        metrics_text += "Result: The Intelligent ANN matches or outperforms standard PID tracking smoothly."
    else:
        metrics_text += "Result: Standard linear PID maintains a rigid edge under low-complexity steady states."

    return fig, metrics_text

# --- Build Gradio Layout ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 MCTA 4362: Intelligent ML Controller Dashboard")
    gr.Markdown("### DC Motor Speed Regulation: Classical PID vs. Artificial Neural Network")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🛠️ Controller Parameters")
            sp_slider = gr.Slider(minimum=10, maximum=100, value=60, step=5, label="Target Speed (Setpoint RPM)")
            dist_time_slider = gr.Slider(minimum=1.0, maximum=4.0, value=2.5, step=0.1, label="Disturbance Injection Time (s)")
            dist_load_slider = gr.Slider(minimum=0.0, maximum=5.0, value=1.5, step=0.1, label="Disturbance Load Torque (Nm)")
            submit_btn = gr.Button("🔄 Run Simulation", variant="primary")

        with gr.Column(scale=2):
            plot_output = gr.Plot(label="System Transient Response")
            metrics_output = gr.Textbox(label="Analytics Summary", lines=5)

    # Link interactions
    submit_btn.click(
        fn=simulate_dashboard,
        inputs=[sp_slider, dist_time_slider, dist_load_slider],
        outputs=[plot_output, metrics_output]
    )

    # Run simulation on load
    demo.load(
        fn=simulate_dashboard,
        inputs=[sp_slider, dist_time_slider, dist_load_slider],
        outputs=[plot_output, metrics_output]
    )

# Launch with a public shareable link enabled
demo.launch(share=True, debug=True)

⏳ Installing stable dashboard environment...
🧠 Modeling plant and pre-training the Intelligent ANN Controller...
🎯 Training complete! Launching Interface...


/tmp/ipykernel_8929/1841289776.py:103: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ac4ffa49ca0505ad55.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ac4ffa49ca0505ad55.gradio.live
